<a href="https://colab.research.google.com/github/PRATHAMESHK24/ML-PARCTICAL.ML/blob/main/Practical7_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# practical7_gaussian_nb.py
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

emails = [
    "win money now", "claim your free prize", "free offer now",
    "win a free gift", "urgent claim prize", "money prize offer",
    "meeting at office", "project meeting tomorrow",
    "please send the report", "lunch with friends",
    "office project report", "see you tomorrow",
    "free money offer", "claim your money",
    "team meeting today", "send project details"
]
labels = np.array([
    1, 1, 1, 1, 1, 1, 0, 0,
    0, 0, 0, 0, 1, 1, 0, 0
])  # 1 = spam, 0 = ham

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails).toarray()

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, random_state=42, stratify=labels
)

# Manual Gaussian Naive Bayes
class ManualGaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.prior = {}

        for c in self.classes:
            data = X[y == c]
            self.mean[c] = data.mean(axis=0)
            self.var[c] = data.var(axis=0) + 1e-9
            self.prior[c] = len(data) / len(y)

    def predict(self, X):
        predictions = []
        for x in X:
            scores = []
            for c in self.classes:
                log_likelihood = -0.5 * np.sum(
                    np.log(2 * np.pi * self.var[c])
                    + (x - self.mean[c]) ** 2 / self.var[c]
                )
                scores.append(np.log(self.prior[c]) + log_likelihood)
            predictions.append(self.classes[np.argmax(scores)])
        return np.array(predictions)

manual = ManualGaussianNB()
manual.fit(X_train, y_train)
manual_pred = manual.predict(X_test)

sk_model = GaussianNB()
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)

print("Manual Gaussian NB accuracy:",
      accuracy_score(y_test, manual_pred))
print("Scikit-learn Gaussian NB accuracy:",
      accuracy_score(y_test, sk_pred))
print("\nScikit-learn classification report:")
print(classification_report(y_test, sk_pred, zero_division=0))

new_emails = ["win a free prize", "project meeting tomorrow"]
new_X = vectorizer.transform(new_emails).toarray()
new_pred = sk_model.predict(new_X)

for email, pred in zip(new_emails, new_pred):
    print(email, "=>", "SPAM" if pred == 1 else "HAM")
